# ARIMA Model

In [ ]:
import pmdarima as pm
import pandas as pd
import joblib

# Dataset configuration to handle different datasets
class DatasetConfig:
    def __init__(self, name, csv_path):
        self.name = name
        self.csv_path = csv_path

Define a ARIMA manager to handle the different type of datasets.

In [ ]:
class ArimaManager:
    def __init__(self, config, scaler_path):
        self.config = config
        self.scaler = joblib.load(scaler_path)
        self.df = pd.read_csv(self.config.csv_path)

    def predict(self, type, predict_length):
        print(f"--- Running ARIMA Forecasting on {len(self.df.columns)} columns ---")
        
        forecasts = {}

        if type == 'D':
            m = 24
        elif type == 'W':
            m = 7
        elif type == 'Y':
            m = 12
        else:
            m = 1

        for col_name in self.df.columns:
            if not pd.api.types.is_numeric_dtype(self.df[col_name]):
                continue
                
            print(f"Fitting ARIMA for: {col_name}...")
            
            try:    
                # Configuration of the arima
                model = pm.auto_arima(
                    self.df[col_name],
                    seasonal=True,
                    m=m,
                    start_p=1, start_q=1,
                    max_p=3, max_q=3,
                    error_action='ignore',
                    suppress_warnings=True,
                    stepwise=True,
                    maxiter=50
                )
                
                preds = model.predict(n_periods=predict_length)
                forecasts[col_name] = preds
            except Exception as e:
                print(f"Warning: ARIMA failed for {col_name} ({e}). using last value fill.")
                last_val = self.df[col_name].iloc[-1]
                forecasts[col_name] = [last_val] * predict_length

        df_forecast = pd.DataFrame(forecasts)
        df_forecast = df_forecast[self.df.columns]
        np_inverse = self.scaler.inverse_transform(df_forecast)
        df_final = pd.DataFrame(np_inverse, columns=self.df.columns)

        output_path = self.config.csv_path.replace(
            "numerical.csv",
            "arima_forecasting.csv"
        )

        df_final.to_csv(output_path, index=False)
        print(f"Done! Saved ARIMA forecasts to: {output_path}")

Generate forecasting data for different datasets.

In [6]:

stock_test_len = len(pd.read_csv("./data/GOOGL_STOCKS/GOOGL_STOCKS_test.csv"))
occupancy_test_len = len(pd.read_csv("./data/OccupancyDetection/OccupancyDetection_test.csv"))
power_test_len = len(pd.read_csv( "./data/PowerConsumption/PowerConsumption_test.csv"))
traffic_test_len = len(pd.read_csv( "./data/TrafficVolume/TrafficVolume_test.csv"))


cfg_stocks = DatasetConfig('GOOGL_STOCKS', "./data/GOOGL_STOCKS/GOOGL_STOCKS_numerical.csv")
arima_stocks = ArimaManager(cfg_stocks, './scaler/GOOGL_STOCKS_scaler.pkl')
arima_stocks.predict('None', stock_test_len)

cfg_occupancy = DatasetConfig('OccupancyDetection', "./data/OccupancyDetection/OccupancyDetection_numerical.csv")
arima_occupancy = ArimaManager(cfg_occupancy, './scaler/OccupancyDetection_scaler.pkl')
arima_occupancy.predict('None', occupancy_test_len)

cfg_power = DatasetConfig('PowerConsumption', "./data/PowerConsumption/PowerConsumption_numerical.csv")
arima_power = ArimaManager(cfg_power, './scaler/PowerConsumption_scaler.pkl')
arima_power.predict('D', power_test_len)

cfg_traffic = DatasetConfig('TrafficVolume', "./data/TrafficVolume/TrafficVolume_numerical.csv")
arima_traffic = ArimaManager(cfg_traffic, './scaler/TrafficVolume_scaler.pkl')
arima_traffic.predict('D', traffic_test_len)



--- Running ARIMA Forecasting on 6 columns ---
Fitting ARIMA for: Open...
Fitting ARIMA for: High...
Fitting ARIMA for: Low...
Fitting ARIMA for: Close...
Fitting ARIMA for: Adj Close...
Fitting ARIMA for: Volume...
Done! Saved ARIMA forecasts to: ./data/GOOGL_STOCKS/GOOGL_STOCKS_arima_forecasting.csv
--- Running ARIMA Forecasting on 6 columns ---
Fitting ARIMA for: Temperature...
Fitting ARIMA for: Humidity...
Fitting ARIMA for: Light...
Fitting ARIMA for: CO2...
Fitting ARIMA for: HumidityRatio...
Fitting ARIMA for: Occupancy...
Done! Saved ARIMA forecasts to: ./data/OccupancyDetection/OccupancyDetection_arima_forecasting.csv
--- Running ARIMA Forecasting on 8 columns ---
Fitting ARIMA for: Temperature...
Fitting ARIMA for: Humidity...
Fitting ARIMA for: Wind_Speed...
Fitting ARIMA for: general_diffuse_flows...
Fitting ARIMA for: diffuse_flows...
Fitting ARIMA for: Zone_1_Power_Consumption...
Fitting ARIMA for: Zone_2_Power_Consumption...
Fitting ARIMA for: Zone_3_Power_Consumption..

c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '


Fitting ARIMA for: snow_1h...


c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '


Fitting ARIMA for: clouds_all...
Fitting ARIMA for: traffic_volume...
Done! Saved ARIMA forecasts to: ./data/TrafficVolume/TrafficVolume_arima_forecasting.csv
